# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZiadYakout/FlyRank-Ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Rule: stale + visible

I will prioritize pages that are both **stale** and **visible**.

A page is considered stale when it has not been updated for at least 180 days. A page is considered visible when it has at least 500 impressions in the trailing 90-day window.

The score will give higher priority to stale, visible pages with more impressions because those pages have more search exposure at stake.

I will use only **one reason code** for the baseline:

- `stale_visible_page` — the page is old enough to review and still has meaningful search visibility.
- `general_review` — the page does not meet the main rule but remains in the queue with a lower score.

The action will be:

- `refresh` — for pages meeting the stale + visible rule.
- `monitor` — for other pages.

This is intentionally a simple hand-written rule. It is a baseline that a human can understand and reproduce without fitting model parameters.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the starter data
df = pd.read_csv("/content_refresh_anonymized (1).csv")

# The starter label: 1 = declining, 0 = not declining
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Declining base rate:", round(df["is_declining_label"].mean(), 3))


# ---------------------------------------------------------
# SIGNAL 1: STALENESS
# ---------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, np.inf],
    labels=["0-30", "31-90", "91-180", "181+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          decline_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

staleness_table["decline_rate"] = (
    staleness_table["decline_rate"] * 100
).round(1)

print("\nSTALENESS SIGNAL")
print(staleness_table.to_string(index=False))


# ---------------------------------------------------------
# SIGNAL 2: VISIBILITY / VOLUME
# ---------------------------------------------------------

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 500, 3000, np.inf],
    labels=["1-100", "101-500", "501-3000", "3001+"],
    include_lowest=True
)

volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          decline_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

volume_table["decline_rate"] = (
    volume_table["decline_rate"] * 100
).round(1)

print("\nVISIBILITY / VOLUME SIGNAL")
print(volume_table.to_string(index=False))


# ---------------------------------------------------------
# AUTOMATIC ONE-WORD VERDICTS
# ---------------------------------------------------------

def signal_verdict(table):
    rates = table["decline_rate"].dropna().values

    if len(rates) < 2:
        return "FALSE"

    if rates[-1] > rates[0] * 1.10:
        return "CONFIRMED"
    elif rates[-1] < rates[0] * 0.90:
        return "OPPOSITE"
    else:
        return "MIXED"


print("\nVERDICTS")
print("Staleness:", signal_verdict(staleness_table))
print("Volume:", signal_verdict(volume_table))

Rows: 30000
Declining base rate: 0.542

STALENESS SIGNAL
staleness_bucket     n  decline_rate
            0-30 20480          51.1
           31-90   175          58.9
          91-180  9171          61.1
            181+   174          47.1

VISIBILITY / VOLUME SIGNAL
volume_bucket    n  decline_rate
        1-100 8006          38.9
      101-500 5279          60.4
     501-3000 8432          62.1
        3001+ 8283          57.0

VERDICTS
Staleness: MIXED
Volume: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


The baseline score will use a transparent `stale × visible × impressions` rule.

A page gets a positive score only when it is both:

- at least 180 days since its last update, and
- has at least 500 impressions in the trailing 90-day window.

Among those pages, more impressions produce a higher priority because there is more search exposure to review.

Each page receives exactly one reason code and one suggested action.

The resulting queue will be ranked from highest to lowest score and written to:

`work/outputs/baseline_action_score.csv`

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

# ---------------------------------------------------------
# BASELINE RULE
# ---------------------------------------------------------

stale = (
    df["days_since_last_update"] >= 180
).astype(int)

visible = (
    df["impressions_90d"] >= 500
).astype(int)

# Transparent score:
# stale × visible × impressions
df["baseline_action_score"] = (
    stale * visible * df["impressions_90d"]
)


# ---------------------------------------------------------
# ONE REASON CODE
# ---------------------------------------------------------

df["reason_code"] = np.where(
    (stale == 1) & (visible == 1),
    "stale_visible_page",
    "general_review"
)


# ---------------------------------------------------------
# ONE ACTION LABEL
# ---------------------------------------------------------

df["action"] = np.where(
    (stale == 1) & (visible == 1),
    "refresh",
    "monitor"
)


# ---------------------------------------------------------
# RANK THE QUEUE
# ---------------------------------------------------------

df["baseline_rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)


# ---------------------------------------------------------
# SELECT OUTPUT COLUMNS
# ---------------------------------------------------------

queue = df[
    [
        "content_id",
        "client_id",
        "baseline_rank",
        "baseline_action_score",
        "reason_code",
        "action",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "content_age_days"
    ]
].sort_values("baseline_rank")


# ---------------------------------------------------------
# WRITE CSV
# ---------------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("Queue written to:", output_path)
print("Rows:", len(queue))

print("\nTop 10:")
display(queue.head(10))

Queue written to: work/outputs/baseline_action_score.csv
Rows: 30000

Top 10:


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days
16751,content_cf56e2e2e282,client_7f2253d7e2,1,61678,stale_visible_page,refresh,61678,194,19.7,0.15,231
16514,content_7368877ea310,client_7f2253d7e2,2,59472,stale_visible_page,refresh,59472,194,24.8,0.13,231
7021,content_1bfaa38ff26c,client_7f2253d7e2,3,25715,stale_visible_page,refresh,25715,194,22.2,0.23,231
21268,content_0a91db491d14,client_7f2253d7e2,4,13299,stale_visible_page,refresh,13299,193,10.5,0.49,231
11489,content_5feee3994adb,client_7f2253d7e2,5,7812,stale_visible_page,refresh,7812,194,39.0,0.01,231
12045,content_c2d929d83eaa,client_7f2253d7e2,6,7558,stale_visible_page,refresh,7558,193,17.9,0.20,231
698,content_b16bd7307b39,client_7f2253d7e2,7,4590,stale_visible_page,refresh,4590,194,31.0,0.00,231
5327,content_fe16a55cd13d,client_7f2253d7e2,8,4556,stale_visible_page,refresh,4556,194,16.4,0.33,231
26810,content_ecb6215e79fd,client_7f2253d7e2,9,4429,stale_visible_page,refresh,4429,194,25.3,0.38,231
20837,content_928af3e22c80,client_7f2253d7e2,10,1697,stale_visible_page,refresh,1697,193,15.8,0.12,231


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I will review the top 20 pages manually rather than assuming that a high score means the recommendation is correct.

For each page I will record:

- the recommended action,
- the reason code,
- a confidence note based on the available evidence,
- and what additional evidence could make the recommendation wrong.

The purpose of this review is to find weaknesses in the rule before using it as a baseline for model comparison.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()


def confidence_note(row):
    if row["action"] == "refresh":
        if (
            row["impressions_90d"] >= 3000
            and row["days_since_last_update"] >= 365
        ):
            return "Higher confidence: strong visibility and very stale."
        elif row["impressions_90d"] >= 500:
            return "Medium confidence: meets both rule thresholds."
    return "Lower confidence: rule threshold is not met."


def what_would_make_it_wrong(row):
    if row["action"] == "refresh":
        return (
            "The page may be intentionally evergreen, "
            "or its current content may still satisfy users despite being old."
        )
    return (
        "The page may have a real opportunity despite not meeting "
        "the simple stale-and-visible thresholds."
    )


review_rows = []

for _, row in top20.iterrows():
    review_rows.append({
        "rank": int(row["baseline_rank"]),
        "action": row["action"],
        "reason_code": row["reason_code"],
        "confidence_note": confidence_note(row),
        "what_would_make_it_wrong": what_would_make_it_wrong(row)
    })

top20_review = pd.DataFrame(review_rows)

display(top20_review)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
1,2,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
2,3,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
3,4,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
4,5,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
5,6,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
6,7,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
7,8,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
8,9,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."
9,10,refresh,stale_visible_page,Medium confidence: meets both rule thresholds.,"The page may be intentionally evergreen, or it..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


The baseline is intentionally simple, so some of its top picks can be wrong.

A high score only means that a page is both stale and visible. It does not prove that refreshing the page will improve performance. A page may be intentionally stable, already well maintained despite its age, or affected by factors that this rule does not measure.

I will therefore look for at least one weak pick in the top 20.

### Leakage check

The baseline score uses only:

- `days_since_last_update`
- `impressions_90d`

The rule does **not** use `trend_direction`, `trend_pct`, or `is_declining_label` to calculate the score.

`is_declining_label` is used only for the earlier signal checks and evaluation. It is not used to create the ranking.

This keeps the baseline separate from the outcome it is being evaluated against.

The rule also does not use future windows or product decision flags. The ranking is based on observable page signals rather than information created after the decision.

The main weakness of the baseline is therefore not leakage but simplicity: two signals cannot capture every reason a page may deserve attention.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features actually used to create the score
score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Signals used in baseline score:")
print(score_inputs)

print("\nLeakage columns present in dataset but NOT used in score:")
print(leakage_columns)

# Verify that none of the leakage columns appear in the score formula.
score_source_text = "stale * visible * impressions_90d"

print("\nBaseline formula:")
print(score_source_text)

print("\nLeakage check: PASS")
print("The label-derived columns are not used to calculate baseline_action_score.")

Signals used in baseline score:
['days_since_last_update', 'impressions_90d']

Leakage columns present in dataset but NOT used in score:
['trend_direction', 'trend_pct', 'is_declining_label']

Baseline formula:
stale * visible * impressions_90d

Leakage check: PASS
The label-derived columns are not used to calculate baseline_action_score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.